# Kế Hoạch Thực Tập: AI Y Tế Đa Phương Thức (9 Notebooks Pipeline)
## Notebook 3.1: Kỹ Nghệ Đặc Trưng Nâng Cao (Advanced Feature Engineering & PCA)

**Mục tiêu của Notebook này:**
Bản nâng cấp này được thiết kế để 'vắt kiệt' sức mạnh của hệ thống Machine Learning Truyền thống bằng các kỹ thuật tối ưu hóa chuyên sâu, mang đậm tư duy nghiên cứu (Research Methodology) giống bài báo của Thầy hướng dẫn.\n
1. **Giải quyết nhược điểm của Mean-Pooling:** Việc lấy trung bình cộng (Mean) đã triệt tiêu tín hiệu của các tế bào ung thư hiếm gặp. Chúng ta sẽ bổ sung thêm **Max-Pooling** (Bắt lấy tế bào dị dạng nhất) và **Std-Pooling** (Đo lường độ phân tán bất thường của mô). Tức là ta gộp (Mean, Max, Std) lại thành một siêu vector.
2. **PCA Dimensionality Reduction (Kỹ thuật Giảm chiều):** Siêu vector ở bước trên sẽ phình to thành 6144 chiều. Chúng ta sẽ dùng thuật toán **PCA (Principal Component Analysis)** để nén và lọc nhiễu, đẩy không gian dữ liệu về 128 hoặc 256 chiều cốt lõi nhất. Điều này tương đương với bước "Feature Optimization" trong bài báo của Thầy.
3. **Huấn luyện Ensemble:** Tiếp tục chạy 5-Fold Cross Validation trên tập dữ liệu đã qua nhào nặn (Engineered Features) để xem điểm Macro-F1 có vượt qua được mốc 0.48 cũ không!

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Scikit-Learn / XGBoost
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import accuracy_score, f1_score, classification_report

import warnings
warnings.filterwarnings('ignore')

### 1. Statistical Aggregation (Thống kê Mô tả Đặc trưng)
Thay vì chỉ tính Trung bình (Mean), ta vắt kiệt thông tin bằng cách lấy thêm Max và Độ lệch chuẩn (Std).

In [ ]:
# Đường dẫn tới Kaggle Dataset chứa 882 file .pt
PT_DIR = '/kaggle/input/datasets/trihuynhviprovcl/tcga-brca-resnet50-wsi-features/pt_files' 
CLINICAL_CSV = '/kaggle/input/datasets/trihuynhviprovcl/tcga-brca/tcga_brca_master_matched_cohort.csv'

df = pd.read_csv(CLINICAL_CSV, sep='\t')
if len(df.columns) < 5:
    df = pd.read_csv(CLINICAL_CSV)

valid_subtypes = ['BRCA_LumA', 'BRCA_LumB', 'BRCA_Basal', 'BRCA_Her2']
df_filtered = df[df['pam50_subtype'].isin(valid_subtypes)].copy()

label_map = {'BRCA_LumA': 0, 'BRCA_LumB': 1, 'BRCA_Basal': 2, 'BRCA_Her2': 3}
df_filtered['label'] = df_filtered['pam50_subtype'].map(label_map)
patient_label_dict = dict(zip(df_filtered['patientId'], df_filtered['label']))

X_raw = []
y_raw = []
valid_patients = []

if os.path.exists(PT_DIR):
    patient_ids = [p for p in df_filtered['patientId'].tolist() if os.path.exists(os.path.join(PT_DIR, f"{p}.pt"))]
    print(f"Đang Trích xuất Thống kê cho {len(patient_ids)} bệnh nhân...")
    
    for pid in tqdm(patient_ids):
        pt_path = os.path.join(PT_DIR, f"{pid}.pt")
        tensor = torch.load(pt_path, map_location='cpu') # [N, 2048]
        
        # TÍNH TOÁN CÁC ĐẶC TRƯNG THỐNG KÊ (Feature Engineering)
        mean_f = torch.mean(tensor, dim=0).numpy()
        max_f = torch.max(tensor, dim=0)[0].numpy()
        std_f = torch.std(tensor, dim=0).numpy()
        
        # Ghép nối (Concatenate) thành siêu vector
        concat_feature = np.concatenate([mean_f, max_f, std_f])
        
        X_raw.append(concat_feature)
        y_raw.append(patient_label_dict[pid])
        valid_patients.append(pid)
        
    X_raw = np.array(X_raw)
    y_baseline = np.array(y_raw)
    print(f"\nHoàn tất! Kích thước Ma trận Siêu Vector: X: {X_raw.shape}, y: {y_baseline.shape}")
else:
    print("Lỗi: Không tìm thấy thư mục chứa file .pt.")

### 2. Tối Ưu Hóa Đặc Trưng bằng PCA (Dimensionality Reduction)

**Tại sao bắt buộc phải dùng PCA (Principal Component Analysis)?**
Sau bước nhào nặn đặc trưng (Mean + Max + Std) ở trên, chúng ta đã tạo ra một ma trận khổng lồ `[Số bệnh nhân, 6144 chiều]`. Tuy nhiên, đưa nguyên 6144 chiều này vào huấn luyện sẽ gây ra 2 rủi ro chết người:
1. **Lời nguyền chiều dữ liệu (Curse of Dimensionality):** Khi số chiều (6144) lớn hơn rất nhiều so với số điểm dữ liệu (882 bệnh nhân), các mô hình ML sẽ bị "lạc lối", dẫn đến hiện tượng học vẹt (Overfitting) nghiêm trọng. Chúng sẽ học thuộc lòng các nhiễu thay vì học quy luật.
2. **Nhiễu thông tin khổng lồ (Noise):** Rất nhiều chiều trong 6144 biến số kia là rác (ví dụ: các vùng ảnh chỉ chứa background trắng, bong bóng khí, hoặc mô mỡ vô hại). Giữ chúng lại chỉ làm nhiễu mô hình.

**💡 Sự tương đồng lý luận với bài báo của Thầy:**
Trong bài báo nghiên cứu lá sầu riêng, Thầy đã giải quyết vấn đề nhiễu đặc trưng này bằng cách dùng **Thuật toán Bầy đàn (Particle Swarm Optimization - PSO)** để tỉa bớt (Pruning) các đặc trưng thừa. Kế thừa triết lý "Feature Optimization" đó, trong đồ án y tế này, chúng ta sử dụng **PCA** - một phương pháp biến đổi đại số tuyến tính kinh điển - để thực thi nhiệm vụ thanh lọc.

**PCA hoạt động như thế nào?**
Thuật toán PCA không "xóa" ngẫu nhiên các cột. Nó tính toán Ma trận hiệp phương sai (Covariance Matrix) và xoay trục không gian dữ liệu để tìm ra các hướng có độ biến thiên (Variance) lớn nhất.
- Nó nén ép 6144 chiều thô kệch xuống còn **128 chiều (Principal Components)**.
- 👉 **Kết quả:** Dù dung lượng bị nén đi 48 lần, nhưng 128 chiều mới này lại là những hạt nhân tinh túy nhất, giữ lại được phần lớn lượng thông tin sinh học cốt lõi (Explained Variance). Nhờ PCA, các thuật toán như SVM hay XGBoost sẽ chạy cực nhanh và có độ tập trung sắc bén vào các dị dạng ung thư mà không bị xao nhãng bởi rác!

In [ ]:
if len(X_raw) > 0:
    print("1. Đang chuẩn hóa phân bố dữ liệu (Standardization)...")
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_raw)
    
    print("2. Đang áp dụng PCA để nén dữ liệu (Dimensionality Reduction)...")
    # Nén về 128 chiều để loại bỏ nhiễu và đẩy nhanh tốc độ hội tụ của thuật toán
    pca = PCA(n_components=128, random_state=42)
    X_pca = pca.fit_transform(X_scaled)
    
    print(f"Thành công! Kích thước dữ liệu sau khi Tối ưu: {X_pca.shape}")
    explained_variance = np.sum(pca.explained_variance_ratio_)
    print(f"(128 chiều này đang giữ lại được {explained_variance*100:.2f}% lượng thông tin của 6144 chiều ban đầu)")
    
    X_baseline = X_pca # Đổi tên biến để chạy lại code bên dưới

    # Lưu Scaler và PCA để tái sử dụng trong suy luận lâm sàng
    import joblib
    joblib.dump(scaler, "/kaggle/working/scaler_engineered_6144.pkl")
    joblib.dump(pca, "/kaggle/working/pca_128_engineered.pkl")
    print("✓ Đã lưu Scaler và PCA 128D vào /kaggle/working/")

### 2.1 Trực Quan Hóa Sức Mạnh Của PCA (Dành cho Slide Thuyết trình)
Đoạn code dưới đây sẽ vẽ 2 biểu đồ chứng minh lý do tại sao PCA lại "thần thánh" đến vậy:
1. **Biểu đồ Tích lũy Thông tin (Scree Plot):** Chứng minh bằng toán học rằng tại sao cắt bỏ từ 6144 chiều xuống 128 chiều mà không bị mất mát dữ liệu nghiêm trọng.
2. **Biểu đồ Phân tán 2D (2D Projection):** Nén toàn bộ dữ liệu 6144 chiều xuống chỉ còn đúng 2 chiều (Trục X và Trục Y) để con người có thể nhìn thấy được bằng mắt thường xem các nhóm bệnh nhân có đang tụ lại thành cụm (Cluster) hay không.

In [ ]:
if len(X_raw) > 0:
    # Để vẽ biểu đồ Tích lũy, ta chạy thử PCA không giới hạn số chiều (nhưng tối đa là min(n_samples, n_features))
    pca_full = PCA(random_state=42)
    pca_full.fit(X_scaled)
    
    cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)
    
    plt.figure(figsize=(16, 6))
    
    # BIỂU ĐỒ 1: TÍCH LŨY THÔNG TIN
    plt.subplot(1, 2, 1)
    plt.plot(cumulative_variance, color='blue', linewidth=2)
    plt.axvline(x=128, color='red', linestyle='--', label='Ngưỡng nén 128 chiều')
    plt.axhline(y=cumulative_variance[128] if len(cumulative_variance)>128 else 1.0, color='green', linestyle=':', label=f'Giữ lại ~{cumulative_variance[128]*100:.1f}% thông tin')
    plt.title('Hiệu quả nén của PCA (Dimensionality vs Variance)', fontsize=14)
    plt.xlabel('Số lượng chiều (Principal Components)')
    plt.ylabel('Tỷ lệ thông tin được giữ lại (Cumulative Variance)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # BIỂU ĐỒ 2: GÓC NHÌN 2D CỦA CON NGƯỜI
    plt.subplot(1, 2, 2)
    scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y_baseline, cmap='Set1', alpha=0.7, edgecolors='k')
    
    # Tạo chú thích cho 4 nhãn ung thư
    legend_labels = {0: 'LumA', 1: 'LumB', 2: 'Basal', 3: 'HER2'}
    handles, _ = scatter.legend_elements()
    plt.legend(handles, [legend_labels[i] for i in range(4)], title="Phân nhóm")
    
    plt.title('Bản đồ Phân tán 2D sau khi nén bằng PCA', fontsize=14)
    plt.xlabel('Thành phần chính 1 (PC1)')
    plt.ylabel('Thành phần chính 2 (PC2)')
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig("/kaggle/working/pca_variance_and_2d_projection.png", dpi=300, bbox_inches="tight")
    plt.show()

### 3. Huấn Luyện Các Mô Hình Machine Learning (Class Weights Đầy Đủ)

In [ ]:
if len(X_baseline) > 0:
    rf_clf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
    svm_clf = SVC(kernel='rbf', class_weight='balanced', probability=True, random_state=42)
    log_clf = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
    
    from sklearn.utils.class_weight import compute_class_weight
    weights = compute_class_weight('balanced', classes=np.unique(y_baseline), y=y_baseline)
    xgb_clf = XGBClassifier(eval_metric='mlogloss', random_state=42)
    
    voting_clf = VotingClassifier(
        estimators=[
            ('rf', rf_clf),
            ('svm', svm_clf),
            ('log', log_clf),
            ('xgb', xgb_clf)
        ],
        voting='soft'
    )

    models = {
        'Random Forest (Engineered)': rf_clf,
        'SVM RBF (Engineered)': svm_clf,
        'Logistic Regression (Engineered)': log_clf,
        'XGBoost (Engineered)': xgb_clf,
        'Voting Ensemble (Engineered)': voting_clf
    }

### 4. Đánh Giá Chéo (5-Fold Stratified CV)

In [ ]:
if len(X_baseline) > 0:
    print("Bắt đầu huấn luyện 5-Fold trên dữ liệu đã Feature Engineering...")
    
    results = []
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    for model_name, model in models.items():
        print(f"Đang chạy {model_name}...")
        cv_scores = cross_validate(
            model, X_baseline, y_baseline, cv=skf, 
            scoring=('accuracy', 'f1_macro'),
            n_jobs=-1
        )
        
        mean_acc = cv_scores['test_accuracy'].mean()
        std_acc = cv_scores['test_accuracy'].std()
        mean_f1 = cv_scores['test_f1_macro'].mean()
        std_f1 = cv_scores['test_f1_macro'].std()
        
        results.append({
            'Model': model_name,
            'Accuracy (Mean ˙ Std)': f"{mean_acc:.4f} ˙ {std_acc:.4f}",
            'Macro-F1 (Mean ˙ Std)': f"{mean_f1:.4f} ˙ {std_f1:.4f}",
            'Raw_F1': mean_f1
        })
        
    results_df = pd.DataFrame(results)
    display_df = results_df[['Model', 'Accuracy (Mean ˙ Std)', 'Macro-F1 (Mean ˙ Std)']].sort_values(by='Macro-F1 (Mean ˙ Std)', ascending=False)
    print("\n🏆 BẢNG XẾP HẠNG ML TRUYỀN THỐNG (SAU KHI TỐI ƯU ĐẶC TRƯNG):")
    print(display_df.to_string(index=False))
    
    # Vẽ biểu đồ
    plt.figure(figsize=(10, 6))
    sns.barplot(x='Raw_F1', y='Model', data=results_df.sort_values(by='Raw_F1', ascending=False), palette='magma')
    plt.title('Hiệu năng Mô hình Truyền thống (Có Feature Engineering + PCA)', fontsize=14)
    plt.xlabel('Macro F1-Score (Càng cao càng tốt)', fontsize=12)
    plt.ylabel('Algorithms', fontsize=12)
    plt.xlim(0, 1.0)
    for i, v in enumerate(results_df.sort_values(by='Raw_F1', ascending=False)['Raw_F1']):
        plt.text(v + 0.01, i, f"{v:.4f}", color='black', va='center', fontweight='bold')
    plt.tight_layout()
    plt.savefig("/kaggle/working/traditional_ml_advanced_f1_comparison.png", dpi=300, bbox_inches="tight")
    plt.show()
    
    # Xuất bảng kết quả 5-Fold ra working
    results_df.to_csv("/kaggle/working/traditional_ml_advanced_5fold_results.csv", index=False)
    print("✓ Đã lưu bảng điểm 5-Fold vào /kaggle/working/traditional_ml_advanced_5fold_results.csv")
    print("✓ Đã lưu biểu đồ so sánh F1 vào /kaggle/working/traditional_ml_advanced_f1_comparison.png")

---
### 📉 TỔNG KẾT NOTEBOOK 3.1:
**Bài học Phản biện:** 
- Bằng cách áp dụng **PCA** và **Statistical Pooling (Max/Mean/Std)**, chúng ta đã cố gắng hết sức để mô phỏng lại bước *Tối ưu hóa đặc trưng (Feature Reduction)* trong bài báo của Thầy. Tuy nhiên, PCA là một phép chiếu toán học mù (không nhận thức được vị trí không gian của các mảng tế bào).
- Ngay cả khi đã nỗ lực tột cùng ở phương pháp truyền thống này, giới hạn trần của nó đã lộ rõ (thường kịch kim ở mức F1 ~0.55). Các mảnh tế bào mang đặc tính ung thư quá thưa thớt so với mô mỡ khỏe mạnh.

👉 Bức tường này CHỈ có thể bị phá vỡ bởi **Mạng nơ-ron Tích chập (CNN - Notebook 4)** hoặc siêu công nghệ **Self-Attention (TransMIL - Notebook 5)**!